In [11]:
import pandas as pd
import numpy as np
import re

INCOME_PATH = "($) CENSUS DATA.csv"
COUNT_PATH  = "(ppl) CENSUS DATA.csv"
OUT_PATH    = "census_clean.csv"

In [12]:
def wide_demographic_sheet_to_long(path: str, value_name: str) -> pd.DataFrame:
    df = pd.read_csv(path)

    # Identify the first column (age labels + the "categories" header row)
    first_col = df.columns[0]

    # Row 0 contains education "categories"
    edu_row = df.iloc[0].copy()

    # Rows 1+ contain the age rows
    data = df.iloc[1:].copy()
    data = data.rename(columns={first_col: "age_mid"})

    # Convert age to numeric
    data["age_mid"] = pd.to_numeric(data["age_mid"], errors="coerce")

    # Build a "group label" for each column: forward-fill race,gender across Unnamed columns
    col_groups = []
    last_group = None
    for c in df.columns[1:]:
        if not str(c).startswith("Unnamed"):
            last_group = str(c).strip()
        col_groups.append(last_group)

    # Education categories aligned with columns 1:
    edu_vals = edu_row[df.columns[1:]].tolist()

    # Create a metadata table for columns
    meta = pd.DataFrame({
        "col": df.columns[1:],
        "group": col_groups,
        "education_years": edu_vals
    })

    # Clean education_years -> numeric
    # Some might be strings like " 11" etc; coerce safely
    meta["education_years"] = (
        meta["education_years"]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
    )
    meta["education_years"] = pd.to_numeric(meta["education_years"], errors="coerce")

    # Split group into race + gender (expects "race, gender")
    # e.g. "white, female"
    meta[["race", "gender"]] = meta["group"].str.split(r"\s*,\s*", n=1, expand=True)

    # Melt data wide -> long
    long = data.melt(
        id_vars=["age_mid"],
        value_vars=df.columns[1:],
        var_name="col",
        value_name=value_name
    )

    # Merge in the metadata (race/gender/education_years)
    long = long.merge(meta[["col", "race", "gender", "education_years"]], on="col", how="left")

    # Clean numeric values (commas, blanks)
    long[value_name] = (
        long[value_name]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.strip()
        .replace({"": np.nan, "nan": np.nan, "None": np.nan})
    )
    long[value_name] = pd.to_numeric(long[value_name], errors="coerce")

    # Drop junk rows (missing key fields)
    long = long.dropna(subset=["age_mid", "education_years", "race", "gender"])

    # Keep only needed columns
    long = long[["race", "gender", "education_years", "age_mid", value_name]]

    return long

In [13]:
income_long = wide_demographic_sheet_to_long(INCOME_PATH, "avg_income")
count_long  = wide_demographic_sheet_to_long(COUNT_PATH,  "population_count")

print("income_long:", income_long.shape)
print("count_long:",  count_long.shape)

income_long.head(50)

income_long: (432, 5)
count_long: (432, 5)


,race,gender,education_years,age_mid,avg_income
0,white,female,8,21,0
1,white,female,8,30,25970
2,white,female,8,40,36690
3,white,female,8,50,29330
4,white,female,8,60,27190
5,white,female,8,73,27290
6,white,female,11,21,12550
7,white,female,11,30,28240
8,white,female,11,40,31750
9,white,female,11,50,32650


In [14]:
cells = income_long.merge(
    count_long,
    on=["race", "gender", "education_years", "age_mid"],
    how="inner"
)

print("merged cells:", cells.shape)
cells.head()

merged cells: (432, 6)


,race,gender,education_years,age_mid,avg_income,population_count
0,white,female,8,21,0,33
1,white,female,8,30,25970,126
2,white,female,8,40,36690,169
3,white,female,8,50,29330,281
4,white,female,8,60,27190,257


In [15]:
cells.to_csv(OUT_PATH, index=False)
OUT_PATH

'census_clean.csv'